# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [13]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
units = df['qty'].sum()
print(df.shape)
print(total_revenue)
print(units)
print(df.head)

(400, 5)
8520.0
783
<bound method NDFrame.head of     vendor_id  category  qty  price  revenue
0        V-10     Drink    2   24.0     48.0
1        V-18  RainGear    1   12.0     12.0
2        V-18     Drink    3    4.5     13.5
3        V-10      Food    2   12.0     24.0
4        V-18     Drink    3    7.5     22.5
..        ...       ...  ...    ...      ...
395      V-18     Merch    1   12.0     12.0
396      V-01     Merch    2   24.0     48.0
397      V-10      Food    3    7.5     22.5
398      V-18     Merch    2   24.0     48.0
399      V-10     Drink    1    7.5      7.5

[400 rows x 5 columns]>


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [24]:
# TODO
by_category = (df.groupby('category')['revenue']
       .sum()
       .sort_values(ascending=False)
       .to_frame('revenue'))
by_category['share_pct'] = (cat['revenue'] / total_revenue * 100).round(2)
by_category

,revenue,share_pct
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [18]:
# TODO
vavg = (df.groupby('vendor_id')['revenue']
        .agg(avg_order='mean', orders='count')
        .sort_values('avg_order', ascending=False)
        .round(2)
)
vavg

,avg_order,orders
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [20]:
# TODO
share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
print("Merch's share of revenue")
print(share)

Merch's share of revenue
20.79225352112676


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [28]:
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(merged) == len(df), f"row count changed: {len(df)} -> {len(merged)}"
assert merged['revenue'].sum() == df['revenue'].sum(), "revenue total changed"
print(f"rows: {len(df)} -> {len(merged)}, revenue: {df['revenue'].sum()} -> {merged['revenue'].sum()}")

unmatched = merged.loc[merged['vendor_name'].isna(), 'vendor_id'].unique()
print(f"unmatched vendor_id(s): {unmatched}")
print(f"orders affected: {merged['vendor_name'].isna().sum()}")
print(f"revenue affected: {merged.loc[merged['vendor_name'].isna(), 'revenue'].sum()}")

rows: 400 -> 400, revenue: 8520.0 -> 8520.0
unmatched vendor_id(s): ['V-18']
orders affected: 108
revenue affected: 2349.0


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [30]:
# TODO
report = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total',
)
report

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [29]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_